In [1]:
import os
import pandas as pd
import sqlite3


while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print("cwd:", os.getcwd())

master = pd.read_csv('data/processed/aqi_weather_master.csv', parse_dates=['date'])
print(master.shape)

cwd: C:\Users\91935\PycharmProjects\AQi project
(15283, 19)


In [2]:
os.makedirs('db', exist_ok=True)
conn = sqlite3.connect('db/aqi.db')

In [3]:
master.to_sql('aqi_weather', conn, if_exists='replace', index=False)

print(pd.read_sql("SELECT COUNT(*) AS n_rows FROM aqi_weather", conn))
print(pd.read_sql("SELECT date FROM aqi_weather LIMIT 3", conn))

   n_rows
0   15283
                  date
0  2015-01-01 00:00:00
1  2015-01-02 00:00:00
2  2015-01-03 00:00:00


In [4]:
q1 = """
WITH monthly AS (
    SELECT
        city,
        strftime('%Y-%m', date)  AS year_month,
        ROUND(AVG(AQI), 1)       AS avg_aqi,
        COUNT(AQI)               AS days_with_aqi
    FROM aqi_weather
    WHERE AQI IS NOT NULL
    GROUP BY city, strftime('%Y-%m', date)
    HAVING COUNT(AQI) >= 20
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY avg_aqi DESC) AS rnk
    FROM monthly
)
SELECT city, rnk AS rank, year_month, avg_aqi, days_with_aqi
FROM ranked
WHERE rnk <= 5
ORDER BY city, rnk;
"""
result1 = pd.read_sql(q1, conn)
print(result1.to_string(index=False))

os.makedirs('sql', exist_ok=True)

def run_and_save(sql_text, filename, comment):
    with open(f'sql/{filename}', 'w', encoding='utf-8') as f:
        f.write(f"-- {comment}\n{sql_text.strip()}\n")
    return pd.read_sql(sql_text, conn)
result1 = run_and_save(q1, '01_worst_months.sql',
                       'Worst 5 months per city by average AQI (months with at least 20 days of AQI data)')

     city  rank year_month  avg_aqi  days_with_aqi
Bengaluru     1    2015-07    160.5             31
Bengaluru     2    2016-03    145.6             31
Bengaluru     3    2016-02    143.9             29
Bengaluru     4    2015-08    134.9             31
Bengaluru     5    2016-01    134.3             31
  Chennai     1    2016-09    184.9             30
  Chennai     2    2015-05    176.1             31
  Chennai     3    2015-04    175.9             29
  Chennai     4    2025-12    166.5             31
  Chennai     5    2015-07    164.1             31
    Delhi     1    2016-11    474.8             30
    Delhi     2    2016-01    441.9             31
    Delhi     3    2016-10    411.1             31
    Delhi     4    2016-12    409.3             31
    Delhi     5    2017-11    406.7             30
  Kolkata     1    2019-01    360.8             31
  Kolkata     2    2018-12    338.3             31
  Kolkata     3    2018-11    283.7             30
  Kolkata     4    2019-02    2

In [6]:
q2 = """
SELECT
    city,
    strftime('%Y', date)                         AS year,
    COUNT(AQI)                                   AS days_with_aqi,
    SUM(CASE WHEN AQI > 300 THEN 1 ELSE 0 END)   AS days_above_300,
    ROUND(100.0 * SUM(CASE WHEN AQI > 300 THEN 1 ELSE 0 END) / COUNT(AQI), 1) AS pct_above_300
FROM aqi_weather
WHERE AQI IS NOT NULL
GROUP BY city, strftime('%Y', date)
ORDER BY city, year;
"""
result2 = pd.read_sql(q2, conn)

result2 = run_and_save(q2, '02_days_above_300.sql',
                       'Days with AQI above 300 per city per year, with percentage of days with data')
print(result2.to_string(index=False))

     city year  days_with_aqi  days_above_300  pct_above_300
Bengaluru 2015            286               1            0.3
Bengaluru 2016            351               3            0.9
Bengaluru 2017            364               0            0.0
Bengaluru 2018            361               1            0.3
Bengaluru 2019            365               0            0.0
Bengaluru 2020            183               0            0.0
Bengaluru 2025            153               0            0.0
Bengaluru 2026            212               0            0.0
  Chennai 2015            276               6            2.2
  Chennai 2016            334               5            1.5
  Chennai 2017            361               5            1.4
  Chennai 2018            365               0            0.0
  Chennai 2019            365               1            0.3
  Chennai 2020            183               0            0.0
  Chennai 2025            153               8            5.2
  Chennai 2026          

In [7]:
def run_and_save(sql_text, filename, comment):
    with open(f'sql/{filename}', 'w', encoding='utf-8') as f:
        f.write(f"-- {comment}\n{sql_text.strip()}\n")
    return pd.read_sql(sql_text, conn)

q3 = """
WITH seasonal AS (
    SELECT
        city,
        CASE
            WHEN CAST(strftime('%m', date) AS INTEGER) IN (11, 12, 1, 2) THEN 'winter'
            WHEN CAST(strftime('%m', date) AS INTEGER) IN (3, 4, 5, 6)   THEN 'summer'
        END AS season_group,
        AQI
    FROM aqi_weather
    WHERE AQI IS NOT NULL
)
SELECT
    city,
    ROUND(AVG(CASE WHEN season_group = 'winter' THEN AQI END), 1) AS winter_avg,
    ROUND(AVG(CASE WHEN season_group = 'summer' THEN AQI END), 1) AS summer_avg,
    ROUND(AVG(CASE WHEN season_group = 'winter' THEN AQI END)
        / AVG(CASE WHEN season_group = 'summer' THEN AQI END), 2) AS winter_to_summer_ratio
FROM seasonal
WHERE season_group IS NOT NULL
GROUP BY city
ORDER BY winter_to_summer_ratio DESC;
"""

result3 = run_and_save(q3, '03_winter_summer_ratio.sql',
                       'Average AQI in winter (Nov-Feb) vs summer (Mar-Jun) per city, and their ratio')
print(result3.to_string(index=False))

     city  winter_avg  summer_avg  winter_to_summer_ratio
  Kolkata       231.1        79.1                    2.92
    Patna       324.2       162.2                    2.00
  Lucknow       309.5       175.4                    1.76
   Mumbai       150.8        89.5                    1.68
    Delhi       353.0       221.6                    1.59
  Chennai       122.9       103.9                    1.18
Bengaluru       104.9        89.1                    1.18


In [4]:
import sys, importlib
import numpy as np
import pandas as pd

sys.path.append('src')
import aqi_calculator
importlib.reload(aqi_calculator)
from aqi_calculator import aqi_and_dominant

m = master.copy()
m['calc_aqi'], m['dominant_pollutant'] = aqi_and_dominant(m)
rec = m['data_source'] == 'openaq_recent'

print("Recent rows with no AQI under the CPCB rule:", m.loc[rec, 'calc_aqi'].isna().sum())

hist = m[(m['data_source'] == 'kaggle_historical') & m['AQI'].notna() & m['calc_aqi'].notna()]
diff = hist['calc_aqi'] - hist['AQI']
print("\nHistorical cross-check under the rule (n =", len(hist), ")")
print("correlation:", round(hist['calc_aqi'].corr(hist['AQI']), 3))
print("mean abs difference:", round(diff.abs().mean(), 1), "| mean difference:", round(diff.mean(), 1))
print("within 10 points:", round((diff.abs() <= 10).mean() * 100, 1), "%")

dom = (m.dropna(subset=['dominant_pollutant'])
         .groupby(['data_source', 'city'])['dominant_pollutant']
         .value_counts(normalize=True).mul(100).round(1).unstack().fillna(0))
print("\nDominant pollutant, % of days (rule applied):")
print(dom)

Recent rows with no AQI under the CPCB rule: 8

Historical cross-check under the rule (n = 10674 )
correlation: 0.942
mean abs difference: 26.8 | mean difference: -10.0
within 10 points: 29.5 %

Dominant pollutant, % of days (rule applied):
dominant_pollutant             CO   NO2   O3  PM10  PM2.5  SO2
data_source       city                                        
kaggle_historical Bengaluru  17.8   0.9  2.1  65.1   14.1  0.0
                  Chennai    14.5   0.2  5.0   5.1   75.2  0.0
                  Delhi       3.4   1.1  0.3  39.5   55.6  0.0
                  Kolkata     8.2   3.5  4.8  42.2   40.4  0.8
                  Lucknow    16.3   0.4  0.5   0.0   82.8  0.0
                  Mumbai     19.1   0.1  0.9  69.0   10.9  0.0
                  Patna       4.4   5.9  4.3   7.2   76.8  1.5
openaq_recent     Bengaluru   3.3   0.6  0.0  92.2    3.9  0.0
                  Chennai     6.3   0.0  0.0  68.6   25.1  0.0
                  Delhi       0.8   1.6  0.0  58.3   39.2  0.0
   

In [5]:
# 1. which recent rows lost their AQI?
print(m.loc[rec & m['calc_aqi'].isna(), ['city', 'date', 'PM2.5', 'PM10', 'O3']])

# 2. extreme CO in the historical track (CO is in mg/m3, so above about 10 is rare)
h = m[m['data_source'] == 'kaggle_historical']
print("\nHistorical CO > 10 mg/m3, rows per city:")
print(h[h['CO'] > 10].groupby('city').size())
print("\nHistorical CO summary by city:")
print(h.groupby('city')['CO'].describe()[['count', 'mean', '50%', 'max']].round(2))

# 3. dominant pollutant again, with CO > 10 treated as invalid
m2 = m.copy()
m2.loc[(m2['data_source'] == 'kaggle_historical') & (m2['CO'] > 10), 'CO'] = np.nan
m2['calc_aqi2'], m2['dominant2'] = aqi_and_dominant(m2)
hh = m2[m2['data_source'] == 'kaggle_historical']
print("\nHistorical dominant pollutant, % of days (CO > 10 set to NaN):")
print(hh.dropna(subset=['dominant2']).groupby('city')['dominant2']
        .value_counts(normalize=True).mul(100).round(1).unstack().fillna(0))

            city       date  PM2.5  PM10         O3
12837  Bengaluru 2025-11-29    NaN   NaN  26.477778
12838  Bengaluru 2025-11-30    NaN   NaN  23.281111
12839  Bengaluru 2025-12-01    NaN   NaN  20.445556
12847  Bengaluru 2025-12-09    NaN   NaN  25.064444
12848  Bengaluru 2025-12-10    NaN   NaN  25.544444
13617      Delhi 2026-01-17  368.0   NaN        NaN
13618      Delhi 2026-01-18  291.0   NaN        NaN
13619      Delhi 2026-01-19  274.0   NaN        NaN

Historical CO > 10 mg/m3, rows per city:
city
Bengaluru    65
Chennai      10
Delhi        65
Lucknow      73
dtype: int64

Historical CO summary by city:
            count  mean   50%    max
city                                
Bengaluru  2009.0  1.84  0.92  48.07
Chennai    1994.0  1.07  0.85  20.62
Delhi      2009.0  1.98  1.24  30.44
Kolkata     814.0  0.80  0.68   3.04
Lucknow    1997.0  2.13  1.37  32.22
Mumbai     1997.0  0.57  0.00   3.65
Patna      1794.0  1.47  1.36   8.06

Historical dominant pollutant, % of days (

In [6]:
import numpy as np
importlib.reload(aqi_calculator)
from aqi_calculator import aqi_and_dominant

master.to_csv('data/processed/aqi_weather_master_v1_backup.csv', index=False)

new = master.copy()
hist = new['data_source'] == 'kaggle_historical'
rec = ~hist
old_recent_aqi = new.loc[rec, 'AQI'].copy()

bad_co = hist & (new['CO'] > 10)
print("Historical CO > 10 set to NaN:", int(bad_co.sum()))
new.loc[bad_co, 'CO'] = np.nan

calc_h, dom_h = aqi_and_dominant(new[hist], min_pollutants=3)
calc_r, dom_r = aqi_and_dominant(new[rec],  min_pollutants=1)

new['calc_aqi'] = np.nan
new['dominant_pollutant'] = np.nan
new.loc[hist, 'calc_aqi'] = calc_h
new.loc[rec,  'calc_aqi'] = calc_r
new.loc[hist, 'dominant_pollutant'] = dom_h
new.loc[rec,  'dominant_pollutant'] = dom_r
new.loc[rec, 'AQI'] = calc_r

changed = ((new.loc[rec, 'AQI'] - old_recent_aqi).abs() > 0.01).sum()
print("Recent rows whose AQI changed:", int(changed))
print("Missing AQI by source:\n", new.groupby('data_source')['AQI'].apply(lambda x: x.isna().sum()))
print("\nRecent rows still without AQI:")
print(new.loc[rec & new['AQI'].isna(), ['city', 'date']].to_string(index=False))
print("\nRows with dominant pollutant by source:\n", new.groupby('data_source')['dominant_pollutant'].count())
print(new.shape)

new.to_csv('data/processed/aqi_weather_master.csv', index=False)
print("Saved corrected master.")

Historical CO > 10 set to NaN: 213
Recent rows whose AQI changed: 58
Missing AQI by source:
 data_source
kaggle_historical    2043
openaq_recent           5
Name: AQI, dtype: int64

Recent rows still without AQI:
     city       date
Bengaluru 2025-11-29
Bengaluru 2025-11-30
Bengaluru 2025-12-01
Bengaluru 2025-12-09
Bengaluru 2025-12-10

Rows with dominant pollutant by source:
 data_source
kaggle_historical    10875
openaq_recent         2561
Name: dominant_pollutant, dtype: int64
(15283, 21)
Saved corrected master.


C:\Users\91935\AppData\Local\Temp\ipykernel_14240\885044152.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... 'CO' 'PM10' 'PM2.5']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  new.loc[hist, 'dominant_pollutant'] = dom_h


In [7]:
import glob, os
import pandas as pd

master = pd.read_csv('data/processed/aqi_weather_master.csv', parse_dates=['date'])
print(master.shape, "| missing AQI:", master['AQI'].isna().sum())

conn = sqlite3.connect('db/aqi.db')
master.to_sql('aqi_weather', conn, if_exists='replace', index=False)
print(pd.read_sql("SELECT COUNT(*) AS n_rows FROM aqi_weather", conn))

results = {}
for path in sorted(glob.glob('sql/0*.sql')):
    with open(path, encoding='utf-8') as f:
        sql_text = f.read()
    results[os.path.basename(path)] = pd.read_sql(sql_text, conn)
    print("ran", os.path.basename(path), "->", len(results[os.path.basename(path)]), "rows")

print("\n--- 01 worst months ---")
print(results['01_worst_months.sql'].to_string(index=False))
print("\n--- 02 days above 300 (2025 and 2026 only) ---")
r2 = results['02_days_above_300.sql']
print(r2[r2['year'].isin(['2025', '2026'])].to_string(index=False))
print("\n--- 03 winter to summer ---")
print(results['03_winter_summer_ratio.sql'].to_string(index=False))
print("\n--- 04 days by category ---")
print(results['04_days_by_category.sql'].to_string(index=False))
if '05_rain_vs_dry_winter.sql' in results:
    print("\n--- 05 rain vs dry ---")
    print(results['05_rain_vs_dry_winter.sql'].to_string(index=False))

(15283, 21) | missing AQI: 2048
   n_rows
0   15283
ran 01_worst_months.sql -> 35 rows
ran 02_days_above_300.sql -> 50 rows
ran 03_winter_summer_ratio.sql -> 7 rows
ran 04_days_by_category.sql -> 41 rows
ran 05_rain_vs_dry_winter.sql -> 7 rows

--- 01 worst months ---
     city  rank year_month  avg_aqi  days_with_aqi
Bengaluru     1    2015-07    160.5             31
Bengaluru     2    2016-03    145.6             31
Bengaluru     3    2016-02    143.9             29
Bengaluru     4    2015-08    134.9             31
Bengaluru     5    2016-01    134.3             31
  Chennai     1    2016-09    184.9             30
  Chennai     2    2015-05    176.1             31
  Chennai     3    2015-04    175.9             29
  Chennai     4    2025-12    166.5             31
  Chennai     5    2015-07    164.1             31
    Delhi     1    2016-11    474.8             30
    Delhi     2    2016-01    441.9             31
    Delhi     3    2016-10    411.1             31
    Delhi     4  